In [ ]:
import torch
import torch.nn as nn
import torch.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Using {} device'.format(device))

block_size = 8
batch_size = 4

: 

In [ ]:
with open('data/wizard_of_oz.txt', 'r', encoding='utf8') as file:
    text = file.read()
    
chars = sorted(set(text))
print(chars)
vocabulary_size = len(chars)
print(vocabulary_size)

: 

In [ ]:
string_to_int = { ch:i for i, ch in enumerate(chars) }
int_to_string = { i:ch for i, ch in enumerate(chars) }

encode = lambda s: [string_to_int[ch] for ch in s]
decode = lambda x: ''.join([int_to_string[i] for i in x])

data = torch.tensor(encode(text), dtype=torch.long)

: 

In [ ]:
# Defines the split (80% train, 20% validation)
n = int(0.8 * len(data))
train_data = data[:n]
validation_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else validation_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

x, y = get_batch('train')

: 

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocabulary_size, vocabulary_size)
        
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            targets = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(index)
            logits = logits[:, -1, :]
            probs = torch.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index
    
model = BigramLanguageModel(vocabulary_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(context)
generated_chars = decode(m.generate(context, max_new_tokens=100)[0].tolist())
print(generated_chars)

: 